# 실습04_사진으로 포즈검출
- MediaPipe Pose를 사용하여 사용자의 현재 자세를 기준 자세(reference pose)와 비교하여 유사도를 평가
- OpenCV를 활용하여 웹캠에서 실시간으로 포즈를 감지하고, 미리 정의된 레퍼런스 이미지의 자세와 비교하여 "Good pose" 또는 "Try pose"를 출력

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

from pathlib import Path
import urllib.request
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
# PoseLandmarker 모델 준비
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

POSE_LANDMARKER_MODEL = MODEL_DIR / 'pose_landmarker_lite.task'
POSE_LANDMARKER_MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task'


def download_model(model_path, model_url):
    if model_path.exists():
        return
    print(f'모델 다운로드 중: {model_path}')
    urllib.request.urlretrieve(model_url, model_path)


download_model(POSE_LANDMARKER_MODEL, POSE_LANDMARKER_MODEL_URL)

In [ ]:
BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
pose_connections = vision.PoseLandmarksConnections
mp_drawing = vision.drawing_utils

drawing_spec1 = mp_drawing.DrawingSpec(thickness=1, circle_radius=1, color=(0, 255, 0))
drawing_spec2 = mp_drawing.DrawingSpec(thickness=4, circle_radius=1, color=(0, 0, 255))

In [ ]:
def create_pose_landmarker(running_mode, num_poses=1):
    # 자세 검출/추적 옵션 설정
    options = PoseLandmarkerOptions(

    )
    return ...


def to_mp_image_bgr(image):
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)


def draw_pose_landmarks(image, pose_result):
    for pose_landmarks in pose_result.pose_landmarks:
        # TODO: 자세 랜드마크/연결선 그리기
        ...


def difference_pose(landmarks1, landmarks2):
    # 랜드마크 평균 거리로 유사도 계산
    # 값이 작을수록 기준 자세와 유사
    threshold = ...  # TODO: 기준값 설정

    if landmarks1 is None or landmarks2 is None:
        return 'Error'

    differences = []
    for lm1, lm2 in zip(landmarks1, landmarks2):
        # TODO: 랜드마크 x, y 거리 계산
        difference = ...
        differences.append(difference)

    result_diff = ...  # TODO: 평균 거리 계산

    # TODO: Good pose / Try pose 판정
    if ...:
        return 'Good pose'
    else:
        return 'Try pose'

In [ ]:
# 기준 이미지 랜드마크 추출
# TODO: 기준 이미지 로드
reference_img_path = ...
ref_img = ...
if ref_img is None:
    raise FileNotFoundError(f'{reference_img_path} 파일을 찾을 수 없습니다.')

ref_landmarks = None
with create_pose_landmarker(VisionRunningMode.IMAGE, num_poses=1) as image_pose_landmarker:
    ref_result = ...  # TODO: 기준 이미지 자세 검출
    if ref_result.pose_landmarks:
        ref_landmarks = ...  # TODO: 기준 랜드마크 저장

if ref_landmarks is None:
    raise RuntimeError('기준 이미지에서 자세를 찾지 못했습니다.')

# 웹캠 자세와 기준 자세 비교
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_pose_landmarker(VisionRunningMode.VIDEO, num_poses=1) as video_pose_landmarker:
    while cap.isOpened():
        res, image = cap.read()
        if not res:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        # 거울 보기용 좌우 반전
        image = cv2.flip(image, 1)
        mp_image = to_mp_image_bgr(image)

        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        result = ...  # TODO: 웹캠 자세 검출

        if result.pose_landmarks:
            # TODO: 자세 랜드마크 그리기
            ...

            # TODO: 첫 번째 사람의 랜드마크 사용
            pose_state = ...
            detect_result = ...  # TODO: 기준 자세와 비교

            # 결과 화면 표시
            cv2.putText(image, detect_result, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)

        cv2.imshow('frame', image)
        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()